# 00 — Context: Frequency Multiplexing Enables Quantum Scale

**Seminar:** Integrated Microcombs for Quantum Applications  
**Speaker:** Xu Yi, University of Virginia

This notebook frames the repository question:

> Which resource scales quantum systems: more devices or more modes?

The goal is to model the architecture distinction that makes integrated microcombs salient:

- **Scale by devices:** \(N\) channels require \(N\) replicated source paths.
- **Scale by modes:** one resonator provides many frequency-indexed quantum channels.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Fallback-safe imports for local repo utilities.
try:
    from src.paths import FIGURES_DIR, RESULTS_DIR, ensure_dir
except Exception:
    FIGURES_DIR = ROOT / "figures"
    RESULTS_DIR = ROOT / "results"
    def ensure_dir(path):
        path.mkdir(parents=True, exist_ok=True)
        return path

try:
    from src.scaling import scaling_table
except Exception:
    def scaling_table(max_channels=100):
        rows = []
        for n in range(1, max_channels + 1):
            rows.append({
                "channels": n,
                "devices_architecture_sources": n,
                "devices_architecture_detectors": n,
                "devices_architecture_optical_paths": n,
                "mode_architecture_resonators": 1,
                "mode_architecture_modes": n,
                "mode_architecture_channels": n,
            })
        return pd.DataFrame(rows)

ensure_dir(FIGURES_DIR)
ensure_dir(RESULTS_DIR / "csv")
ensure_dir(RESULTS_DIR / "json")

FIGURES_DIR, RESULTS_DIR

## Seminar context

Integrated microcombs use an ultra-high-Q optical microresonator to generate many optical frequency modes on a photonic chip.

For quantum applications, those frequency modes can function as parallel quantum channels.  
The architectural question is whether scaling primarily requires additional physical devices or additional addressable frequency modes.

In [ ]:
comparison = pd.DataFrame([
    {
        "architecture": "Scale by devices",
        "source": "N independent sources",
        "resource": "hardware replication",
        "channel_rule": "N channels → N source paths",
        "repo_question": "How costly is replicated hardware?"
    },
    {
        "architecture": "Scale by modes",
        "source": "1 integrated microresonator",
        "resource": "frequency multiplexing",
        "channel_rule": "N channels → N frequency modes",
        "repo_question": "How far can mode count carry scaling?"
    },
])

comparison

In [ ]:
comparison_path = RESULTS_DIR / "csv" / "00_architecture_comparison.csv"
comparison.to_csv(comparison_path, index=False)

summary = {
    "notebook": "00_context",
    "question": "Which resource scales quantum systems: more devices or more modes?",
    "outputs": [
        "figures/00_devices_vs_modes.png",
        "figures/00_frequency_modes.png",
        "results/csv/00_architecture_comparison.csv",
        "results/json/00_context_summary.json",
    ],
}

summary_path = RESULTS_DIR / "json" / "00_context_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))

comparison_path, summary_path

## Scaling sketch

This first model is intentionally simple.

For \(N\) channels:

- device-scaled architecture uses \(N\) source paths;
- mode-scaled architecture uses one resonator plus \(N\) frequency modes.

This does not claim loss, addressability, detection, or fabrication are solved.  
It only isolates the resource substitution that makes microcomb architectures interesting.

In [ ]:
df = scaling_table(max_channels=100)

# Add a fallback optical paths column if the src version does not include it yet.
if "devices_architecture_optical_paths" not in df.columns:
    df["devices_architecture_optical_paths"] = df["channels"]

if "mode_architecture_channels" not in df.columns:
    df["mode_architecture_channels"] = df["channels"]

df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    df["channels"],
    df["devices_architecture_sources"],
    label="Scale by devices: sources",
    linewidth=2,
)
ax.plot(
    df["channels"],
    df["mode_architecture_resonators"],
    label="Scale by modes: resonators",
    linewidth=2,
)
ax.plot(
    df["channels"],
    df["mode_architecture_modes"],
    label="Scale by modes: frequency modes",
    linewidth=2,
    linestyle="--",
)

ax.set_title("Scaling by Devices vs Scaling by Modes")
ax.set_xlabel("Quantum channels")
ax.set_ylabel("Resource count")
ax.legend()
ax.grid(True, alpha=0.3)

fig.tight_layout()

figure_path = FIGURES_DIR / "00_devices_vs_modes.png"
fig.savefig(figure_path, dpi=200)
plt.show()

figure_path

## Frequency-mode sketch

A microcomb supplies a ladder of frequency modes:

\[
f_n = f_0 + n\Delta f
\]

Symmetric modes around the pump can be paired:

\[
(-n, +n)
\]

Later notebooks use this structure to build pair graphs, channel maps, and multipartite entanglement network sketches.

In [ ]:
modes = np.arange(-10, 11)

fig, ax = plt.subplots(figsize=(10, 3.5))

ax.vlines(modes, 0, 1, linewidth=1)
ax.scatter(modes, np.ones_like(modes), s=28)

ax.axvline(0, linestyle="--", alpha=0.6)
ax.text(0, 1.12, "pump", ha="center", va="bottom")

# Draw a few symmetric pair brackets.
for idx, n in enumerate(range(1, 6)):
    y = 0.72 - idx * 0.1
    ax.plot([-n, n], [y, y], linewidth=1)
    ax.scatter([-n, n], [y, y], s=12)
    ax.text(0, y - 0.055, f"(-{n}, +{n})", ha="center", fontsize=8)

ax.set_title("Frequency-Multiplexed Quantum Modes")
ax.set_xlabel("Mode index n")
ax.set_yticks([])
ax.set_ylim(0, 1.25)
ax.set_xlim(modes.min() - 1, modes.max() + 1)

fig.tight_layout()

figure_path = FIGURES_DIR / "00_frequency_modes.png"
fig.savefig(figure_path, dpi=200)
plt.show()

figure_path

## Takeaway

The architectural shift is:

**Scale by devices**

\[
N\ \text{channels} \rightarrow N\ \text{source paths}
\]

**Scale by modes**

\[
1\ \text{resonator} \rightarrow N\ \text{frequency modes} \rightarrow N\ \text{channels}
\]

This motivates the repo question:

> Which resource scales quantum systems more effectively: additional hardware devices or additional frequency modes?